# face-anonymizer — Colab 데모

YOLO-FaceV2 + ByteTrack 기반 영상 얼굴 비식별화(모자이크).

순서: GPU 확인 → 저장소 클론 → 설치 → 가중치 → 영상 업로드 → 실행 → 미리보기 → 다운로드

⚠️ **런타임 → 런타임 유형 변경 → GPU** 먼저 켜세요.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. 저장소 클론

`<YOUR_REPO_URL>` 을 본인 GitHub 저장소 주소로 바꾸세요.

In [ ]:
# 본인 저장소로 교체
REPO_URL = "https://github.com/<YOUR_ID>/face-anonymizer.git"
!git clone -q $REPO_URL
%cd face-anonymizer

## 2. 설치 (ffmpeg 는 Colab 기본 내장)

In [ ]:
!pip install -q -r requirements.txt
import torch, cv2, supervision as sv
print("torch", torch.__version__, "| opencv", cv2.__version__, "| supervision", sv.__version__)

## 3. YOLO-FaceV2 리포 + 가중치 준비 (한 번만)

In [ ]:
!python setup_weights.py

## 4. 입력 영상 업로드

In [ ]:
from google.colab import files
up = files.upload()
INPUT = next(iter(up))
print("input:", INPUT)

## 5. 실행

작은 얼굴 놓치면 `--imgsz 1280 --conf 0.15`, 더 강하게 가리려면 `--mosaic-scale 0.05`.

In [ ]:
OUTPUT = "output_anon.mp4"
!python -m face_anonymizer.cli "$INPUT" -o "$OUTPUT" --method mosaic --imgsz 960 --conf 0.25 --mosaic-scale 0.06 --pad 0.15 --linger 5

## 6. 미리보기 (원본 vs 결과)

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
def grab(path, n=3):
    cap=cv2.VideoCapture(path); tot=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); o=[]
    for f in np.linspace(0,max(0,tot-1),n).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f)); ok,im=cap.read()
        if ok: o.append(cv2.cvtColor(im,cv2.COLOR_BGR2RGB))
    cap.release(); return o
orig,anon=grab(INPUT),grab(OUTPUT); n=min(len(orig),len(anon))
plt.figure(figsize=(12,4*n))
for i in range(n):
    plt.subplot(n,2,2*i+1); plt.imshow(orig[i]); plt.title("original"); plt.axis("off")
    plt.subplot(n,2,2*i+2); plt.imshow(anon[i]); plt.title("anonymized"); plt.axis("off")
plt.tight_layout(); plt.show()

## 7. 다운로드

In [ ]:
from google.colab import files
files.download(OUTPUT)